# Experimento 06 — Conversão e validação Keras → TensorFlow Lite FP16

Objetivos:
1. carregar `melhor_modelo_exp06.keras`;
2. converter para TFLite com pesos FP16;
3. comparar tamanhos FP32 × FP16;
4. validar numericamente Keras FP32 × TFLite FP16;
5. comparar TFLite FP32 × FP16 quando a saída anterior estiver disponível;
6. gerar os artefatos de referência para a Raspberry Pi Zero 2 W.


In [ ]:
import os
import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)


In [ ]:
KERAS_MODEL_PATH = "melhor_modelo_exp06.keras"
TFLITE_FP32_PATH = "melhor_modelo_exp06.tflite"
TFLITE_FP16_PATH = "melhor_modelo_exp06_fp16.tflite"

BENCHMARK_INPUT_PATH = "benchmark_input.npy"
KERAS_OUTPUT_PATH = "benchmark_output_keras.npy"
TFLITE_FP32_OUTPUT_PATH = "benchmark_output_tflite.npy"
TFLITE_FP16_OUTPUT_PATH = "benchmark_output_tflite_fp16.npy"

assert os.path.exists(KERAS_MODEL_PATH), f"Arquivo não encontrado: {KERAS_MODEL_PATH}


## Carregamento do modelo Keras


In [ ]:
model = tf.keras.models.load_model(KERAS_MODEL_PATH, compile=False)
model.summary()


## Conversão FP16

A otimização mantém a interface de entrada/saída em `float32`, mas reduz para `float16` os pesos elegíveis armazenados no arquivo TFLite.


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_fp16_model = converter.convert()

with open(TFLITE_FP16_PATH, "wb") as f:
    f.write(tflite_fp16_model)

print("Modelo salvo:", TFLITE_FP16_PATH)


## Comparação de tamanho


In [ ]:
def kib(path):
    return os.path.getsize(path) / 1024

print(f"Keras:       {kib(KERAS_MODEL_PATH):.2f} KiB")
print(f"TFLite FP16: {kib(TFLITE_FP16_PATH):.2f} KiB")

if os.path.exists(TFLITE_FP32_PATH):
    fp32 = kib(TFLITE_FP32_PATH)
    fp16 = kib(TFLITE_FP16_PATH)
    print(f"TFLite FP32: {fp32:.2f} KiB")
    print(f"Redução:     {100*(1-fp16/fp32):.2f}%")


## Reutilização da mesma entrada de benchmark


In [ ]:
assert os.path.exists(BENCHMARK_INPUT_PATH), (
    "benchmark_input.npy não encontrado. "
    "Use o mesmo arquivo da validação FP32 para preservar a comparabilidade."
)

entrada = np.load(BENCHMARK_INPUT_PATH).astype(np.float32)
print("Entrada:", entrada.shape, entrada.dtype)


## Saída Keras FP32 de referência


In [ ]:
saida_keras = model(entrada, training=False).numpy().astype(np.float32)
np.save(KERAS_OUTPUT_PATH, saida_keras)
print("Saída Keras:", saida_keras.shape, saida_keras.dtype)


## Inferência TFLite FP16 no PC


In [ ]:
interpreter = tf.lite.Interpreter(model_path=TFLITE_FP16_PATH)
inp = interpreter.get_input_details()

interpreter.resize_tensor_input(inp[0]["index"], entrada.shape, strict=False)
interpreter.allocate_tensors()

inp = interpreter.get_input_details()
out = interpreter.get_output_details()

print("Entrada:", inp)
print("\nSaída:", out)

interpreter.set_tensor(inp[0]["index"], entrada)
interpreter.invoke()
saida_fp16 = interpreter.get_tensor(out[0]["index"]).astype(np.float32)

np.save(TFLITE_FP16_OUTPUT_PATH, saida_fp16)
print("\nSaída FP16 salva:", TFLITE_FP16_OUTPUT_PATH)


## Equivalência numérica

Diferenças em relação ao Keras FP32 são esperadas, pois os pesos elegíveis foram armazenados em menor precisão.


In [ ]:
def comparar(ref, teste, titulo):
    d = teste.astype(np.float64) - ref.astype(np.float64)
    a = np.abs(d)
    print(f"--- {titulo} ---")
    print(f"Máxima diferença absoluta: {np.max(a):.10e}")
    print(f"Média diferença absoluta:  {np.mean(a):.10e}")
    print(f"RMSE entre as saídas:      {np.sqrt(np.mean(d**2)):.10e}")
    for tol in (1e-5, 1e-4, 1e-3):
        print(f"allclose ({tol:.0e}):", np.allclose(teste, ref, rtol=tol, atol=tol))

comparar(saida_keras, saida_fp16, "Keras FP32 × TFLite FP16")


## Comparação TFLite FP32 × TFLite FP16


In [ ]:
if os.path.exists(TFLITE_FP32_OUTPUT_PATH):
    saida_fp32 = np.load(TFLITE_FP32_OUTPUT_PATH).astype(np.float32)
    comparar(saida_fp32, saida_fp16, "TFLite FP32 × TFLite FP16")
else:
    print("benchmark_output_tflite.npy não encontrado; comparação ignorada.")


## Artefatos para a Raspberry Pi Zero 2 W

Envie:
- `melhor_modelo_exp06_fp16.tflite`
- `benchmark_input.npy`
- `benchmark_output_tflite_fp16.npy`

Primeiro validaremos PC × Raspberry com o mesmo modelo FP16. Depois repetiremos o protocolo de 10 warm-ups + 100 inferências para comparar diretamente com a baseline FP32.
